# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hapepaAhmed/my-capstone-project/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import pandas as pd
import numpy as np

from google.colab import userdata
from huggingface_hub import hf_hub_download

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


In [2]:
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN
)

print("March data downloaded:")
print(march_path)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

March data downloaded:
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [3]:
queue_columns = [
    "content_hash_id",
    "client_hash_id",
    "report_date",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec"
]

queue_data = pd.read_parquet(
    march_path,
    columns=queue_columns
)

print("Shape:", queue_data.shape)
print("Date range:", queue_data["report_date"].min(), "to", queue_data["report_date"].max())
print("\nColumns:")
print(queue_data.columns.tolist())

Shape: (9841378, 9)
Date range: 2026-03-01 to 2026-03-31

Columns:
['content_hash_id', 'client_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'ga4_total_engagement_sec']


In [4]:
# Build content-level signals for the March action queue

queue_data["ctr"] = np.where(
    queue_data["gsc_impressions"] > 0,
    queue_data["gsc_clicks"] / queue_data["gsc_impressions"],
    np.nan
)

queue_data["engagement_rate"] = np.where(
    queue_data["ga4_sessions"] > 0,
    queue_data["ga4_engaged_sessions"] / queue_data["ga4_sessions"],
    np.nan
)

queue_data["avg_engagement_sec"] = np.where(
    queue_data["ga4_sessions"] > 0,
    queue_data["ga4_total_engagement_sec"] / queue_data["ga4_sessions"],
    np.nan
)

# Aggregate daily observations to content-client level
queue = (
    queue_data
    .groupby(
        ["content_hash_id", "client_hash_id"],
        as_index=False
    )
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        sessions=("ga4_sessions", "sum"),
        engaged_sessions=("ga4_engaged_sessions", "sum"),
        total_engagement_sec=("ga4_total_engagement_sec", "sum")
    )
)

# Recalculate monthly rates from aggregated totals
queue["ctr"] = np.where(
    queue["impressions"] > 0,
    queue["clicks"] / queue["impressions"],
    np.nan
)

queue["engagement_rate"] = np.where(
    queue["sessions"] > 0,
    queue["engaged_sessions"] / queue["sessions"],
    np.nan
)

queue["avg_engagement_sec"] = np.where(
    queue["sessions"] > 0,
    queue["total_engagement_sec"] / queue["sessions"],
    np.nan
)

print("Queue shape:", queue.shape)
print("\nMissing values:")
print(
    queue[
        [
            "impressions",
            "ctr",
            "avg_position",
            "engagement_rate",
            "avg_engagement_sec"
        ]
    ].isna().sum()
)

Queue shape: (331437, 11)

Missing values:
impressions                0
ctr                   154699
avg_position          154699
engagement_rate       241200
avg_engagement_sec    241200
dtype: int64


In [5]:
# Keep content with enough search exposure for an actionable queue

queue = queue[
    queue["impressions"] >= 10
].copy()

print("Rows after impressions >= 10:", len(queue))
print()
print("Missing values:")
print(
    queue[
        [
            "ctr",
            "avg_position",
            "engagement_rate",
            "avg_engagement_sec"
        ]
    ].isna().sum()
)

Rows after impressions >= 10: 143206

Missing values:
ctr                       0
avg_position              0
engagement_rate       77273
avg_engagement_sec    77273
dtype: int64


In [6]:
# Inspect the distributions used for the action queue

print("CTR percentiles:")
print(
    queue["ctr"].quantile(
        [0.25, 0.50, 0.75, 0.90]
    )
)

print("\nAverage position percentiles:")
print(
    queue["avg_position"].quantile(
        [0.25, 0.50, 0.75, 0.90]
    )
)

print("\nImpressions percentiles:")
print(
    queue["impressions"].quantile(
        [0.25, 0.50, 0.75, 0.90]
    )
)

CTR percentiles:
0.25    0.000000
0.50    0.000000
0.75    0.002883
0.90    0.006949
Name: ctr, dtype: float64

Average position percentiles:
0.25     5.337441
0.50     9.281279
0.75    21.608828
0.90    39.978553
Name: avg_position, dtype: float64

Impressions percentiles:
0.25      76.0
0.50     339.0
0.75    1507.0
0.90    4935.5
Name: impressions, dtype: float64


In [10]:
# Fast vectorized priority score + reason codes

ctr_q25 = queue["ctr"].quantile(0.25)
position_q75 = queue["avg_position"].quantile(0.75)
impressions_median = queue["impressions"].median()

# Priority score
queue["priority_score"] = (
    (queue["ctr"] <= ctr_q25).astype(int) * 2
    + (
        (queue["avg_position"] > 10) &
        (queue["avg_position"] <= position_q75)
    ).astype(int) * 2
    + (queue["impressions"] >= impressions_median).astype(int)
)

# Reason codes
queue["reason_code"] = np.select(
    [
        (queue["ctr"] <= ctr_q25) &
        (queue["avg_position"] > 10) &
        (queue["avg_position"] <= position_q75),

        (queue["ctr"] <= ctr_q25),

        (queue["avg_position"] > 10) &
        (queue["avg_position"] <= position_q75),

        (queue["impressions"] >= impressions_median)
    ],
    [
        "Low CTR + search-position opportunity",
        "Low CTR despite search impressions",
        "Search-position opportunity",
        "Meaningful search exposure"
    ],
    default="Lower-priority review"
)

queue = queue.sort_values(
    ["priority_score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

print("Priority score distribution:")
print(queue["priority_score"].value_counts().sort_index())

print("\nTop 10 action items:")
print(
    queue[
        [
            "content_hash_id",
            "client_hash_id",
            "impressions",
            "ctr",
            "avg_position",
            "priority_score",
            "reason_code"
        ]
    ].head(10).to_string(index=False)
)

Priority score distribution:
priority_score
0     9318
1    44219
2    49060
3    22864
4    13224
5     4521
Name: count, dtype: int64

Top 10 action items:
         content_hash_id          client_hash_id  impressions  ctr  avg_position  priority_score                           reason_code
content_70d9d7d2c814b431 client_23a62021009f63c4        23997  0.0     20.845697               5 Low CTR + search-position opportunity
content_17d994b99d470434 client_62f4a7e64f5e0096        19938  0.0     10.300517               5 Low CTR + search-position opportunity
content_cc9732f0da1d8d2d client_fef1a8f436438636        17115  0.0     13.318344               5 Low CTR + search-position opportunity
content_551a522809e0358c client_fef1a8f436438636        11355  0.0     16.918878               5 Low CTR + search-position opportunity
content_b154f6c2652cfeb9 client_73cda7b4e4f265ea        11344  0.0     12.520681               5 Low CTR + search-position opportunity
content_155bbd621ff9af82 client_

### How to read the queue

The queue is a rule-based prioritization layer built from March 2026 observed search and engagement signals. It is not a replacement for the predictive model and does not claim that the listed changes will improve performance.

Priority is based on three observable signals:

- **CTR:** lower observed CTR receives higher priority.
- **Search position:** positions above 10 but within the upper quartile of the observed position distribution are treated as a review opportunity.
- **Impressions:** higher search exposure increases priority because the signal is based on a larger amount of observed search activity.

The highest-priority items combine low CTR, a search-position opportunity, and meaningful search exposure. In the current March data, the 25th percentile of CTR is 0, so the low-CTR rule primarily identifies content with zero recorded clicks despite having at least 10 impressions.

The queue should be interpreted as a **human-review shortlist**, not as an automated recommendation to modify content. Before taking action, a reviewer should inspect the actual page, search queries, intent, title/snippet, and other relevant context.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended use and limits

### Intended use

The ranked queue is designed to help an SEO or content team **prioritize which content-client pairs should be reviewed first**.

It uses observed March 2026 search-performance signals to identify items that deserve human attention. The queue is intended for **triage and decision support**, not automatic content optimization.

The intended workflow is:

1. Use the queue to identify high-priority items.
2. Open the corresponding content and inspect its actual search context.
3. Review queries, search intent, title/snippet, position, and recent changes.
4. Decide manually whether an action is appropriate.
5. Monitor the result after any change.

### Limits

The queue has several important limitations:

- It is based on **March 2026 observed data**, so the priorities can change as search behavior and content performance change.
- The minimum threshold of **10 impressions** removes very-low-exposure observations from the queue.
- The rules identify **associations in observed data**, not causes. A low CTR does not prove that a page needs a title, content, or SEO change.
- The queue does not contain page-level semantic information such as the actual URL, title, search queries, or content quality, so the ranked list cannot determine the correct action by itself.
- GA4 engagement signals are missing for many observations and should not be interpreted as zero engagement.
- The queue is designed for the observed content-client population and should be recalculated when the data period or population changes.
- Priority scores are relative to the current March distribution. They should not be treated as universal performance thresholds.

### Appropriate interpretation

The output should therefore be described as:

> **A transparent prioritization mechanism for human review based on observed search-performance signals.**

It should not be described as an automated system that determines which content should be changed or as evidence that a particular intervention will improve performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human review + the no-go list

### Human review checklist

A high-priority queue item should be reviewed by a person before any action is taken.

The reviewer should check:

- **Actual page:** Open the content and verify that the page is accessible, indexed, and still relevant.
- **Search queries:** Check which queries generated the impressions and whether they match the page's intended topic.
- **Search intent:** Determine whether the page satisfies the intent behind the queries.
- **Title and snippet:** Check whether the search result accurately represents the page and gives users a clear reason to click.
- **Search position:** Consider whether the observed position provides a meaningful opportunity for improvement.
- **Recent changes:** Check whether the page was recently created, updated, redirected, or otherwise changed.
- **Seasonality:** Consider whether the observed performance is affected by seasonal or temporary search behavior.
- **Tracking quality:** Verify that impressions, clicks, and engagement measurements are available and behaving as expected.
- **Content relationships:** Check for duplicate, competing, canonical, or overlapping pages before recommending changes.

### No-go list

The queue should **never automatically**:

- Rewrite or publish page content.
- Change titles or meta descriptions.
- Delete, redirect, or canonicalize pages.
- Infer that low CTR is caused by poor content quality.
- Treat missing engagement data as zero engagement.
- Recommend changes without checking the underlying search queries and page context.
- Suppress or remove content solely because it has a low priority score.
- Treat the priority score as a prediction of future performance.
- Claim that an intervention caused an improvement without an appropriate evaluation.

### Human decision boundary

The system determines **what deserves attention first**.

The human reviewer determines **whether an action is appropriate and what that action should be**.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 4. Monitoring / retrain triggers

### Monitoring the ranked queue

Because the current queue is rule-based rather than a deployed predictive model, the main requirement is to **recalculate and monitor the rules and their input distributions** as new data becomes available.

The queue should be reviewed when:

- A new reporting period becomes available.
- The distribution of CTR, impressions, or average position changes substantially.
- The proportion of high-priority items changes substantially from one period to another.
- Missingness in GA4 or GSC measurements increases.
- Search tracking or warehouse pipelines are changed.
- New clients or substantially different content populations are introduced.
- A known data-quality or tracking issue occurs.

### Threshold recalibration

The current thresholds are derived from the March 2026 distribution, including the CTR 25th percentile, position 75th percentile, and median impressions.

These thresholds should therefore be **recomputed for a new evaluation period** rather than treated as permanent business rules.

Changes in the resulting queue should be monitored to distinguish:

- genuine changes in content performance,
- changes in the underlying content population,
- changes in search behavior,
- and changes caused by data or tracking problems.

### If the predictive model is reintroduced

If the ranking model is used again to generate recommendations, model performance should also be monitored on future data.

Potential retraining or investigation triggers include:

- declining NDCG on a future-period evaluation set,
- increasing MAE or RMSE,
- deteriorating or negative R²,
- substantial changes in feature distributions,
- increased missingness in important features,
- changes in the relationship between search signals and CTR,
- or evidence that the model's ranking quality has degraded for important client or content groups.

Any retraining should be evaluated using a **time-aware validation design** so that future information is not used to construct the training features or targets.

### Monitoring principle

The queue should be treated as a system that requires periodic **recalculation, validation, and human review**, rather than as a static set of recommendations.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [11]:
from pathlib import Path

# Create output directory
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Add an explicit rank
queue_export = queue.copy()
queue_export.insert(0, "rank", np.arange(1, len(queue_export) + 1))

# Select columns useful for the paper / human review
export_columns = [
    "rank",
    "content_hash_id",
    "client_hash_id",
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "sessions",
    "engaged_sessions",
    "total_engagement_sec",
    "engagement_rate",
    "avg_engagement_sec",
    "priority_score",
    "reason_code"
]

queue_export[export_columns].to_csv(
    output_dir / "ranked_action_queue.csv",
    index=False
)

# Save a smaller paper-ready shortlist
queue_export[export_columns].head(100).to_csv(
    output_dir / "ranked_action_queue_top100.csv",
    index=False
)

# Save priority-score summary
priority_summary = (
    queue["priority_score"]
    .value_counts()
    .sort_index()
    .rename_axis("priority_score")
    .reset_index(name="count")
)

priority_summary["percentage"] = (
    priority_summary["count"] / len(queue) * 100
)

priority_summary.to_csv(
    output_dir / "priority_score_summary.csv",
    index=False
)

print("Exports created:")
for path in sorted(output_dir.iterdir()):
    print(f"- {path}")


Exports created:
- work/outputs/priority_score_summary.csv
- work/outputs/ranked_action_queue.csv
- work/outputs/ranked_action_queue_top100.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.